# Extractor Agent — Document Extraction with Nova 2 Lite

The **Extractor Agent** is the second specialist in the Phase 1 process for insurance claims processing. After the Authenticator verifies identity and coverage, the Extractor's job is to pull structured data out of every document submitted with the claim. 

**Note:** In this notebook, we will set up the Extractor to receive documents directly in its request payload. In production, these document references arrive from the Supervisor Agent, which owns S3 access and routes documents to specialist agents. 

### Document Extraction with Nova 2 Lite - The JSON Schema Pattern
A good practice for document extraction with Nova 2 Lite is to define a formal JSON Schema for each document type it will analyze, and include the schema in the extraction prompt. This tells Nova exactly what fields to extract, what types they should be, and which are required. Nova then returns a JSON object that conforms to the schema.


## Setup

In [ ]:
import boto3
import json
import sys
import os
from pathlib import Path
from strands import Agent, tool
from strands.models import BedrockModel
from IPython.display import IFrame
from IPython.display import display, Image
from typing import List
import base64

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT       = Path("..").resolve()
SAMPLE_DATA_PATH = REPO_ROOT / "sample_data"


# ── Model configuration ───────────────────────────────────────────────────────
REGION   = "us-east-1"
MODEL_ID = "us.amazon.nova-2-lite-v1:0"

model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    max_tokens=4096,   
    additional_request_fields={
        "reasoningConfig": {
            "type": "enabled",
            "maxReasoningEffort": "medium"
        }
    }
)

# ── Direct Bedrock Runtime client — used inside @tool extraction functions ────
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

print(f"✅ Nova 2 Lite configured with reasoning enabled")
print(f"   Model           : {MODEL_ID}")
print(f"   Sample data path: {SAMPLE_DATA_PATH}")



## Part A: View the Source Documents

Before running any extraction, let's look at the actual documents the Extractor Agent will process. These are the sample claim files stored in `insurance-claims-processing/sample_data/`.

Viewing the raw documents first gives you a QA baseline — after extraction, you can compare the agent's structured output field-by-field against what you see in the PDF.

The seven documents below represent a complete life insurance death claim submission:

| Document | File | Purpose in Claims Processing |
|---|---|---|
| Death Certificate | death_certificate_sample.pdf | Official record of death + cause |
| Insurance Policy | policy_whole_life_sample.pdf | Coverage details + beneficiary designation |
| Medical Records | medical_records_sample.pdf | Patient history + conditions |
| Will Document | will_document_sample.pdf | Asset distribution + executor |
| Trust Document | trust_document_sample.pdf | Trust structure + beneficiaries |
| Beneficiary ID | beneficiary_id_sample.pdf | Claimant identity verification |
| Police Report | police_report_sample.pdf | Death investigation + foul play assessment |

> **Note for SageMaker Studio:** If IFrame doesn't render inline (some Studio environments block local file rendering), the PDFs are viewable directly in the file browser at `insurance-claims-processing/sample_data/`. The extraction will work regardless of whether the viewer renders.


In [ ]:

from IPython.display import IFrame, display, HTML

SAMPLE_DOCUMENTS = {
    "death_certificate": "death_certificate_sample.pdf",
    "policy_document":   "policy_whole_life_sample.pdf",
    "medical_records":   "medical_records_sample.pdf",
    "will_document":     "will_document_sample.pdf",
    "trust_document":    "trust_document_sample.pdf",
    "beneficiary_id":    "beneficiary_id_sample.pdf",
    "police_report":     "police_report_sample.pdf",
}

In [ ]:
# ===============================================================================
#Define the file you want to see using the keys in the SAMPLE_DOCUMENTS variable
#Change document key below to view the document
#You can also go to the sample_data directory to view each file
# ===============================================================================



filepath = SAMPLE_DATA_PATH / SAMPLE_DOCUMENTS["policy_document"]

with open(filepath, "rb") as f:
        pdf_data = base64.b64encode(f.read()).decode('utf-8')
        
# Display as embedded PDF
display(HTML(f'''
    <iframe 
        src="data:application/pdf;base64,{pdf_data}" 
        width="600" 
        height="900"
        type="application/pdf">
    </iframe>
'''))

## Part B: Build the agent tool for document extraction


### 1. Define document schemas
Each document type has a JSON schema that defines the exact fields Nova should extract. The schema is embedded directly in the prompt — Nova reads the PDF and returns a JSON object conforming to the schema. Nova 2 Lite uses these schemas as output contracts to return data in a consistent, typed structure


In [ ]:

# =============================================================================
# JSON SCHEMAS — strictly derived from actual document field names
# =============================================================================

DEATH_CERTIFICATE_SCHEMA = {
    "type": "object",
    "description": "Structured data extracted from a death certificate",
    "properties": {
        "header": {
            "type": "object",
            "description": "Certificate identification fields from the document header",
            "properties": {
                "state_file_no": {"type": "string"},
                "county": {"type": "string"},
                "certificate_no": {"type": "string"},
                "date_filed": {"type": "string", "format": "date"}
            }
        },
        "decedent_information": {
            "type": "object",
            "description": "Personal information about the deceased",
            "properties": {
                "name_last_first": {"type": "string"},
                "sex": {"type": "string"},
                "dob": {"type": "string", "format": "date"},
                "age": {"type": "integer"},
                "ssn_masked": {"type": "string"},
                "marital": {"type": "string"},
                "residence": {"type": "string"},
                "birthplace": {"type": "string"}
            }
        },
        "death_information": {
            "type": "object",
            "description": "Details about the death event",
            "properties": {
                "date_of_death": {"type": "string", "format": "date"},
                "time": {"type": "string"},
                "place_of_death": {"type": "string"},
        	    "city_co_st": {"type": "string"},
                "location": {"type": "string"},
                "in_hospice": {"type": "string"},
                "manner": {"type": "string"},
                "autopsy": {"type": "string"}
            }
        },
        "cause_of_death": {
            "type": "object",
            "description": "Cause of death chain as recorded on the certificate",
            "properties": {
                "immediate_cause_a": {"type": "string"},
                "interval_a": {"type": "string"},
                "due_to_b": {"type": "string"},
                "interval_b": {"type": "string"},
                "due_to_c": {"type": "string"},
                "interval_c": {"type": "string"},
                "other_conditions": {"type": "string"}
            }
        },
        "informant": {
            "type": "object",
            "description": "Person who provided information for the certificate",
            "properties": {
                "name": {"type": "string"},
                "relationship": {"type": "string"},
                "mailing_address": {"type": "string"},
                "phone": {"type": "string"}
            }
        },
        "certifier": {
            "type": "object",
            "description": "Physician who certified the death",
            "properties": {
                "physician": {"type": "string"},
                "license_no": {"type": "string"},
                "facility": {"type": "string"},
                "phone": {"type": "string"},
                "date_signed": {"type": "string", "format": "date"}
            }
        }
    },
    "required": ["decedent_information", "death_information", "cause_of_death", "certifier"]
}

POLICY_DOCUMENT_SCHEMA = {
    "type": "object",
    "description": "Structured data extracted from a whole life insurance policy",
    "properties": {
        "policy_overview": {
            "type": "object",
            "description": "Core policy identification and status fields",
            "properties": {
                "policy_number": {"type": "string"},
                "plan": {"type": "string"},
                "issue_date": {"type": "string", "format": "date"},
                "status": {"type": "string"},
                "face_amount": {"type": "number"},
                "premium_mode": {"type": "string"},
                "annual_premium": {"type": "number"},
                "paid_to_date": {"type": "string", "format": "date"}
            }
        },
        "insured_owner": {
            "type": "object",
            "description": "Insured and policy owner information",
            "properties": {
                "name": {"type": "string"},
                "date_of_birth": {"type": "string", "format": "date"},
                "ssn_masked": {"type": "string"},
                "address": {"type": "string"},
                "phone": {"type": "string"},
                "email": {"type": "string"}
            }
        },
        "beneficiary_designation": {
            "type": "object",
            "description": "Primary beneficiary designation details",
            "properties": {
                "primary_beneficiary": {"type": "string"},
                "relationship": {"type": "string"},
                "benefit_percent": {"type": "number"},
                "beneficiary_address": {"type": "string"},
                "beneficiary_phone": {"type": "string"},
                "beneficiary_email": {"type": "string"}
            }
        },
        "policy_values": {
            "type": "object",
            "description": "Current policy financial values",
            "properties": {
                "cash_surrender_value": {"type": "number"},
                "outstanding_policy_loan": {"type": "number"},
                "net_death_benefit_estimated": {"type": "number"}
            }
        },
        "customer_service": {
            "type": "object",
            "description": "Carrier contact information",
            "properties": {
                "carrier_name": {"type": "string"},
                "carrier_address": {"type": "string"},
                "phone": {"type": "string"},
                "claims_email": {"type": "string"}
            }
        }
    },
    "required": ["policy_overview", "insured_owner", "beneficiary_designation", "policy_values"]
}

MEDICAL_RECORDS_SCHEMA = {
    "type": "object",
    "description": "Structured data extracted from a medical records continuity of care summary",
    "properties": {
        "patient_information": {
            "type": "object",
            "description": "Patient demographics and provider assignment",
            "properties": {
                "patient_name": {"type": "string"},
                "mrn": {"type": "string"},
                "date_of_birth": {"type": "string", "format": "date"},
                "sex": {"type": "string"},
                "primary_address": {"type": "string"},
                "phone": {"type": "string"},
                "primary_care_provider": {"type": "string"},
                "npi": {"type": "string"}
            }
        },
        "problem_list_active": {
            "type": "array",
            "description": "Active medical conditions from the problem list",
            "items": {"type": "string"}
        },
        "medication_list": {
            "type": "array",
            "description": "Current medications",
            "items": {
                "type": "object",
                "properties": {
                    "medication": {"type": "string"},
                    "dose_route": {"type": "string"},
                    "frequency": {"type": "string"},
                    "indication": {"type": "string"}
                }
            }
        },
        "recent_encounter": {
            "type": "object",
            "description": "Most recent clinical encounter details",
            "properties": {
                "encounter_type": {"type": "string"},
                "date": {"type": "string", "format": "date"},
                "location": {"type": "string"},
                "assessment_and_plan": {"type": "string"}
            }
        },
        "laboratory_results": {
            "type": "array",
            "description": "Recent lab results",
            "items": {
                "type": "object",
                "properties": {
                    "test": {"type": "string"},
                    "date": {"type": "string", "format": "date"},
                    "result": {"type": "string"},
                    "units": {"type": "string"},
                    "reference_range": {"type": "string"}
                }
            }
        },
        "provider_attestation": {
            "type": "object",
            "description": "Physician attestation details",
            "properties": {
                "date": {"type": "string", "format": "date"},
                "printed_name": {"type": "string"},
                "title": {"type": "string"}
            }
        }
    },
    "required": ["patient_information", "problem_list_active", "medication_list"]
}

WILL_DOCUMENT_SCHEMA = {
    "type": "object",
    "description": "Structured data extracted from a Last Will and Testament",
    "properties": {
        "testator": {
            "type": "object",
            "description": "Identity and residence of the person making the will",
            "properties": {
                "name": {"type": "string"},
                "date_of_birth": {"type": "string", "format": "date"},
                "residence": {"type": "string"},
                "marital_status": {"type": "string"}
            }
        },
        "family_information": {
            "type": "object",
            "description": "Family status and living descendants",
            "properties": {
                "marital_status": {"type": "string"},
                "living_descendants": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "relationship": {"type": "string"}
                        }
                    }
                }
            }
        },
        "executor": {
            "type": "object",
            "description": "Appointed executor of the estate",
            "properties": {
                "name": {"type": "string"},
                "role": {"type": "string"},
                "alternate": {"type": "string"}
            }
        },
        "specific_bequests": {
            "type": "array",
            "description": "Specific gifts named in the will",
            "items": {
                "type": "object",
                "properties": {
                    "item": {"type": "string"},
                    "beneficiary": {"type": "string"}
                }
            }
        },
        "residuary_estate": {
            "type": "object",
            "description": "Disposition of the residuary estate",
            "properties": {
                "beneficiary": {"type": "string"},
                "description": {"type": "string"}
            }
        },
        "insurance_proceeds": {
            "type": "object",
            "description": "Treatment of life insurance proceeds under the will",
            "properties": {
                "policy_numbers_referenced": {
                    "type": "array",
                    "items": {"type": "string"}
                },
                "instruction": {"type": "string"}
            }
        },
        "execution": {
            "type": "object",
            "description": "Signing, witness, and notarization details",
            "properties": {
                "date_signed": {"type": "string", "format": "date"},
                "governing_law": {"type": "string"},
                "witnesses": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "address": {"type": "string"}
                        }
                    }
                },
                "notary_public": {"type": "string"},
                "commission_expires": {"type": "string", "format": "date"},
                "notarized_date": {"type": "string", "format": "date"},
                "notarized_county": {"type": "string"}
            }
        }
    },
    "required": ["testator", "executor", "residuary_estate", "insurance_proceeds"]
}

TRUST_DOCUMENT_SCHEMA = {
    "type": "object",
    "description": "Structured data extracted from a Revocable Living Trust Agreement",
    "properties": {
        "trust_name": {"type": "string"},
        "agreement_date": {"type": "string", "format": "date"},
        "grantor": {"type": "string"},
        "trustees": {
            "type": "object",
            "description": "Initial and successor trustee designations",
            "properties": {
                "initial_trustee": {"type": "string"},
                "successor_trustee": {"type": "string"},
                "successor_relationship": {"type": "string"}
            }
        },
        "revocability": {"type": "string"},
        "beneficiaries": {
            "type": "object",
            "description": "Beneficiary designations during lifetime and upon death",
            "properties": {
                "lifetime_beneficiary": {"type": "string"},
                "remainder_beneficiaries": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "percent": {"type": "number"}
                        }
                    }
                }
            }
        },
        "distribution_upon_death": {
            "type": "array",
            "description": "Ordered distribution instructions upon grantor's death",
            "items": {"type": "string"}
        },
        "trustee_powers": {
            "type": "array",
            "description": "Enumerated powers granted to the trustee",
            "items": {"type": "string"}
        },
        "schedule_a_initial_property": {
            "type": "array",
            "description": "Initial property transferred to the trust per Schedule A",
            "items": {
                "type": "object",
                "properties": {
                    "description": {"type": "string"},
                    "notes": {"type": "string"}
                }
            }
        },
        "execution": {
            "type": "object",
            "description": "Signing and notarization details",
            "properties": {
                "date_signed": {"type": "string", "format": "date"},
                "grantor_printed_name": {"type": "string"},
                "notary_public": {"type": "string"},
                "commission_expires": {"type": "string", "format": "date"},
                "notarized_date": {"type": "string", "format": "date"},
                "notarized_county": {"type": "string"}
            }
        }
    },
    "required": ["trust_name", "agreement_date", "grantor", "trustees", "beneficiaries"]
}

BENEFICIARY_ID_SCHEMA = {
    "type": "object",
    "description": "Structured data extracted from a government-issued photo ID card",
    "properties": {
        "issuing_state": {"type": "string"},
        "document_type": {"type": "string"},
        "last_name": {"type": "string"},
        "first_name": {"type": "string"},
        "dob": {"type": "string", "format": "date"},
        "sex": {"type": "string"},
        "eyes": {"type": "string"},
        "ht": {"type": "string"},
        "wt": {"type": "integer"},
        "dl_id_number": {"type": "string"},
        "iss": {"type": "string", "format": "date"},
        "exp": {"type": "string", "format": "date"},
        "address": {"type": "string"}
    },
    "required": ["last_name", "first_name", "dob", "dl_id_number", "exp", "address"]
}

POLICE_REPORT_SCHEMA = {
    "type": "object",
    "description": "Structured data extracted from a police incident report",
    "properties": {
        "agency": {
            "type": "object",
            "description": "Law enforcement agency details",
            "properties": {
                "agency_name": {"type": "string"},
                "ori": {"type": "string"}
            }
        },
        "case_information": {
            "type": "object",
            "description": "Case identification and classification",
            "properties": {
                "case_no": {"type": "string"},
                "report_date": {"type": "string", "format": "date"},
                "incident_type": {"type": "string"},
                "disposition": {"type": "string"}
            }
        },
        "incident_details": {
            "type": "object",
            "description": "Time, location, and responding unit details",
            "properties": {
                "date_time": {"type": "string"},
                "location": {"type": "string"},
                "call_type": {"type": "string"},
                "units": {
                    "type": "array",
                    "items": {"type": "string"}
                },
                "reporting_officer": {"type": "string"},
                "supervisor": {"type": "string"}
            }
        },
        "persons_involved": {
            "type": "array",
            "description": "All persons documented in the report",
            "items": {
                "type": "object",
                "properties": {
                    "role": {"type": "string"},
                    "name": {"type": "string"},
                    "dob": {"type": "string", "format": "date"},
                    "contact": {"type": "string"},
                    "notes": {"type": "string"}
                }
            }
        },
        "narrative": {"type": "string"},
        "evidence_property": {
            "type": "object",
            "description": "Evidence collected and property inventoried",
            "properties": {
                "evidence_collected": {"type": "string"},
                "property_inventoried": {"type": "string"}
            }
        },
        "officer_certification": {
            "type": "object",
            "description": "Officer signature and certification details",
            "properties": {
                "date": {"type": "string", "format": "date"},
                "printed_name_badge": {"type": "string"},
                "unit": {"type": "string"}
            }
        }
    },
    "required": ["case_information", "incident_details", "persons_involved", "narrative"]
}

# ── Schema registry ───────────────────────────────────────────────────────────
SCHEMA_REGISTRY = {
    "death_certificate": DEATH_CERTIFICATE_SCHEMA,
    "policy_document":   POLICY_DOCUMENT_SCHEMA,
    "medical_records":   MEDICAL_RECORDS_SCHEMA,
    "will_document":     WILL_DOCUMENT_SCHEMA,
    "trust_document":    TRUST_DOCUMENT_SCHEMA,
    "beneficiary_id":    BENEFICIARY_ID_SCHEMA,
    "police_report":     POLICE_REPORT_SCHEMA,
}

In [ ]:
print("✅ SCHEMA_REGISTRY defined for all document types:")
for doc_type, schema in SCHEMA_REGISTRY.items():
    required = schema.get("required", [])
    n_fields = len(schema.get("properties", {}))
    print(f"   {doc_type:<25} fields: {n_fields:<4} required: {required}")

### 2. Define helper functions
We define two helper functions that will be used when we define the document processing tool for the agent


In [ ]:
# =============================================================================
# Define a helper function to which maps each file type to a `TOOL_REGISTRY` key 
# =============================================================================

def classify_document(file_path: str, doc_type_hint: str = None) -> str:
    """
    Maps a document filename to a SCHEMA_REGISTRY key using keyword heuristics.
    Accepts an optional doc_type_hint to bypass heuristics when the Supervisor
    Agent passes an explicit document type label alongside the file path.
    Returns 'unknown' if no match is found.
    """
    if doc_type_hint:
        return doc_type_hint

    filename = Path(file_path).name.lower()

    if "death" in filename:
        return "death_certificate"
    elif "policy" in filename or "whole_life" in filename:
        return "policy_document"
    elif "medical" in filename or "records" in filename:
        return "medical_records"
    elif "will" in filename or "testament" in filename:
        return "will_document"
    elif "trust" in filename:
        return "trust_document"
    elif "beneficiary_id" in filename or "license" in filename:
        return "beneficiary_id"
    elif "police" in filename or "incident" in filename or "report" in filename:
        return "police_report"
    else:
        return "unknown"


# Quick classification test
print("🔍 Classification test:")
for doc_type, filename in SAMPLE_DOCUMENTS.items():
    classified = classify_document(str(SAMPLE_DATA_PATH / filename))
    match      = "✅" if classified == doc_type else f"⚠️  got '{classified}'"
    print(f"   {filename:<45} → {classified:<25} {match}")


In [ ]:
# =============================================================================
# Define a helper function `that performs a document extraction call. 
# It reads DF bytes, embeds the document's JSON schema in the prompt, and 
# calls Nova 2 Lite via `bedrock_runtime.converse`. 
# =============================================================================



def _call_nova_for_extraction(file_path: str, doc_type: str, schema: dict) -> dict:
    """Internal helper: call Nova 2 Lite once for a single document extraction.

    Reads raw PDF bytes and passes them to Nova as a native document content
    block. The JSON schema for the document type is embedded in the prompt —
    Nova returns a plain JSON object conforming to the schema.

    Called once per document inside process_claim_documents. Not intended to
    be called directly.
    """
    # ── Read PDF bytes ────────────────────────────────────────────────────────
    with open(file_path, "rb") as f:
        pdf_bytes = f.read()

    # ── Build schema string for prompt injection ──────────────────────────────
    properties = schema.get("properties", schema)
    schema_str = json.dumps(properties, indent=2)

    # ── Build extraction prompt ───────────────────────────────────────────────
    doc_label = doc_type.replace("_", " ").title()
    prompt = (
        f"You are a document extraction specialist processing a {doc_label}.\n"
        "Extract all information from this document and return ONLY a valid JSON object "
        "with the following fields:\n"
        f"{schema_str}\n"
        "Rules:\n"
        "- Use null for any field you cannot find in the document\n"
        "- Do not guess or hallucinate values\n"
        "- Return dates in YYYY-MM-DD format\n"
        "- Return ONLY the JSON object — no explanation, no markdown fences, no extra text"
    )

    # ── Call Nova 2 Lite via bedrock_runtime.converse ─────────────────────────
    response = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{
            "role": "user",
            "content": [
                {
                    "document": {
                        "format": "pdf",
                        "name": doc_type,
                        "source": {"bytes": pdf_bytes}
                    }
                },
                {"text": prompt}
            ]
        }],
        additionalModelRequestFields={
            "reasoningConfig": {
                "type": "enabled",
                "maxReasoningEffort": "medium"
            }
        },
        inferenceConfig={"maxTokens": 4096}
    )

    # ── Extract text from Nova's response ─────────────────────────────────────
    output_text = ""
    content_blocks = response.get("output", {}).get("message", {}).get("content", [])
    for block in content_blocks:
        if not isinstance(block, dict):
            continue
        if block.get("text", "").strip():
            output_text = block["text"].strip()
            break
    # ── Strip accidental markdown fences if Nova includes them ────────────────
    if output_text.startswith("```"):
        lines = output_text.splitlines()
        cleaned_lines = [line for line in lines
                         if not line.strip().startswith("```")]
        output_text = "\n".join(cleaned_lines).strip()

    # ── Parse JSON ────────────────────────────────────────────────────────────
    try:
        return json.loads(output_text)
    except json.JSONDecodeError:
        # Last resort: try to find a JSON object anywhere in the response
        json_match = re.search(r'\{[\s\S]+\}', output_text)
        if json_match:
            try:
                return json.loads(json_match.group())
            except json.JSONDecodeError:
                pass

    return {
        "error":        "JSON parse failed — Nova did not return valid JSON",
        "raw_response": output_text[:500]
    }


In [ ]:
# =============================================================================
# Build the Extractor agent's tool for extraction using existing helper functions
# =============================================================================

@tool
def process_claim_documents(document_paths: List[str]) -> str:
    """
    Extract structured data from all documents in a claim submission.

    Accepts a list of PDF file paths — one per document in the claim.
    For each document:
      - Classifies the document type from the filename using classify_document()
      - Selects the correct JSON schema from SCHEMA_REGISTRY
      - Calls Nova 2 Lite once per document via bedrock_runtime.converse
        with the schema embedded in the prompt
      - Parses Nova's plain JSON text response
      - Validates that all required fields are present
      - Writes the extracted JSON to a file in the same directory as the source PDF

    Returns a consolidated JSON string with:
      - extraction_results: dict keyed by document type with extracted fields
      - validation_summary: dict keyed by document type with pass/fail + missing fields
      - ready_for_downstream: bool — true only if all required extractions passed

    Args:
        document_paths: List of file path strings, one per claim document
    """
    extraction_results = {}
    validation_summary = {}
    all_passed         = True

    print(f"📋 Processing {len(document_paths)} documents...")
    print()

    for i, file_path in enumerate(document_paths, 1):
        path = Path(file_path)
        print(f"── Document {i}/{len(document_paths)}: {path.name}")

        # Step 1: Classify document type from filename
        doc_type = classify_document(file_path)
        if doc_type == "unknown":
            print(f"   ⚠️  Could not classify — skipping")
            extraction_results[path.name] = {"error": f"Unknown document type: {path.name}"}
            validation_summary[path.name] = {"passed": False, "missing_fields": [], "warnings": ["Unknown document type"]}
            all_passed = False
            continue

        print(f"   Type     : {doc_type}")

        # Step 2: Select schema from registry
        schema = SCHEMA_REGISTRY.get(doc_type)
        if not schema:
            print(f"   ❌ No schema found for {doc_type}")
            extraction_results[doc_type] = {"error": f"No schema for: {doc_type}"}
            validation_summary[doc_type] = {"passed": False, "missing_fields": [], "warnings": ["No schema registered"]}
            all_passed = False
            continue

        # Step 3: Verify file exists
        if not path.exists():
            print(f"   ❌ File not found: {file_path}")
            extraction_results[doc_type] = {"error": f"File not found: {file_path}"}
            validation_summary[doc_type] = {"passed": False, "missing_fields": schema.get("required", []), "warnings": ["File not found"]}
            all_passed = False
            continue

        # Step 4: Call Nova once for this document
        print(f"   Calling Nova  : schema embedded in prompt")
        try:
            extracted = _call_nova_for_extraction(
                file_path=str(path),
                doc_type=doc_type,
                schema=schema
            )
        except Exception as e:
            print(f"   ❌ Nova call failed: {e}")
            extracted = {"error": str(e)}

        extraction_results[doc_type] = extracted

        # Step 5: Write extracted JSON to notebook working directory
        output_file = Path.cwd() / f"02_integrate_agent_tools"/f"{path.stem}_extracted.json"
        try:
            with open(output_file, "w") as f:
                json.dump(extracted, f, indent=2)
            print(f"   💾 Saved       : {output_file.name}  ← look for this file in your notebook directory")
        except Exception as e:
            print(f"   ❌ Write failed: {type(e).__name__}: {e}")


        # Step 6: Validate required fields
        required = schema.get("required", [])
        if "error" in extracted:
            missing  = required
            warnings = [f"Extraction failed: {extracted['error']}"]
            passed   = False
        else:
            missing  = [f for f in required if extracted.get(f) is None]
            warnings = [f"Required field '{f}' is null" for f in required if f in extracted and extracted[f] is None]
            passed   = len(missing) == 0

        validation_summary[doc_type] = {
            "passed":         passed,
            "missing_fields": missing,
            "warnings":       warnings
        }

        if not passed:
            all_passed = False

        status = "✅ passed" if passed else f"⚠️  missing: {missing}"
        print(f"   Validation    : {status}")
        print()

    consolidated = {
        "extraction_results":   extraction_results,
        "validation_summary":   validation_summary,
        "ready_for_downstream": all_passed
    }

    print(f"── Extraction complete. ready_for_downstream: {all_passed}")
    return json.dumps(consolidated, indent=2)




## Build and run the Extraction agent

In [ ]:
EXTRACTOR_SYSTEM_PROMPT = """
You are the Extractor Agent in a life insurance claims processing pipeline.

Your job is to extract structured data from all documents in a claim submission.

## How to Respond

When you receive a claim document manifest (a list of file paths), call the
process_claim_documents tool ith the complete list of file paths.

## Output

Include the full consolidated JSON from the tool result in your response.
"""

extractor_agent = Agent(
    model=model,
    system_prompt=EXTRACTOR_SYSTEM_PROMPT,
    tools=[process_claim_documents]
)

In [ ]:
# -----------------------------------------------------------------------------
# Run the Extractor Agent
# -----------------------------------------------------------------------------


document_manifest = [
    str(SAMPLE_DATA_PATH / filename)
    for filename in SAMPLE_DOCUMENTS.values()
]

paths_str = "\n".join(f"- {p}" for p in document_manifest)

EXTRACTION_REQUEST = f"""Please extract structured data from the following claim documents:

{paths_str}

Call the process_claim_documents tool with all file paths and return the consolidated results."""

print("🚀 Running Extractor Agent...")
print(f"   Documents in manifest : {len(document_manifest)}")
for p in document_manifest:
    print(f"   └ {Path(p).name}")


extraction_response = extractor_agent(EXTRACTION_REQUEST)

print()
print("✅ Extraction complete")


## ✅ Notebook Complete: Extractor Agent

### What You Built

- **Extractor Agent** with a document extraction tool via @tool decorator and defined function 
- Document processing of a diverse set of documents using Amazon Nova 2 Lite
  
### Key Patterns Used
| Pattern | Description |
|---|---|
| **Single-tool agent** | Concentrating all processing logic inside one tool keeps the agent's role narrow — receive input, call tool, report result. Reduces agent reasoning overhead and makes behavior deterministic and testable. |
| **Registry-based dispatch** | A central registry maps input classifiers (document type, event type, entity type) to configuration objects (schemas, tool specs, handlers). Decouples classification logic from processing logic and makes the system extensible without code changes. |
| **Inline validation at extraction** | Required field validation runs immediately after each extraction call, before results are aggregated. Failures are surfaced per-record rather than at the end of a batch, enabling partial success and targeted retry. |
| **Structured handoff payload** | Each agent produces a standardized output object (`extraction_results`, `validation_summary`, `ready_for_downstream`) that the next agent consumes without transformation. Enables agent substitution and parallel pipeline development. |

### Limitations of This Notebook

| Limitation | Why It Exists | What Addresses It |
|---|---|---|
| Documents loaded from local disk | Simulates Supervisor handoff | Supervisor Agent owns S3 retrieval |
| Sequential extraction | Documents processed one at a time inside the tool | Parallel extraction via async patterns |
| No cross-document validation | Each document extracted independently | NB06: Policy Verification Agent cross-references fields |
| No claim state persistence | Results exist only in memory and local files | NB06: DynamoDB claim record updated with extraction results |
| Filename-based classification only | Heuristic matching — won't handle arbitrary filenames | Production: Supervisor passes explicit doc type labels |
